# Getting started with Dataset

In [1]:
import numpy as np
import IPython
import matplotlib.pyplot as plt
import torch
import torchaudio

import megamicros
from megamicros.log import log
from megamicros.ailab.dataset import AidbDataset, AidbTimeStrechingTransform

log.setLevel( "WARNING" )
#log.setLevel( "INFO" )
print( f"megamicros-{megamicros.__version__}" )


megamicros-2.0.53


In [2]:
# Plot spectrogram

def plot_specgram(waveform, sample_rate, title="Spectrogram"):
    waveform = waveform.numpy()

    figure, ax = plt.subplots()
    ax.specgram(waveform[0], Fs=sample_rate)
    figure.suptitle(title)
    figure.tight_layout()

In [3]:
# Set database access credentials
DBHOST = 'http://dbwelfare.biimea.io/'
#DBHOST = 'http://dbchantier.biimea.io/'
LOGIN = 'ailab'
EMAIL = 'bruno.gas@biimea.com'
PASSWORD = '#T;uZnQ5UJ_JC~&'

DATASET = 'test0'
#LABELS = 'sow-grunt-mod-stress'
#LABELS = (1, 2, 3)

split_size = 2.0
audio_transforms = torch.nn.Sequential(
    #torchaudio.transforms.MelSpectrogram(sample_rate=10000, n_mels=128),
    torchaudio.transforms.Spectrogram( n_fft=400, win_length=400, hop_length=160, pad=0, power=2, normalized=False ),
    AidbTimeStrechingTransform( factor=0.5, output_length=int( split_size*10000 ) ),
    #torchaudio.transforms.FrequencyMasking(freq_mask_param=15),
    #torchaudio.transforms.TimeMasking(time_mask_param=35)
)

dataset = AidbDataset( '/Users/brunogas/Data/dataset/assets', DBHOST, DATASET, channels=[1,], split_size=split_size, zero_padding=True, transform=audio_transforms )

print( f"Number of extracted samples in database: {len( dataset )}" )

# extract data and print their shape
#for data_idx, [data, sample_rate, label] in enumerate( dataset ):
#    print( f"Extracted [{data_idx}] signal (label:{label}, sr={sample_rate}) of shape: {np.shape( data )}" )

0%
0%
0%
0%
20%
20%
20%
40%
40%
40%
60%
60%
60%
80%
80%
80%
Number of extracted samples in database: 5


In [ ]:
import os
import torchaudio
_SAMPLE_DIR = "_assets"
YESNO_DATASET_PATH = os.path.join(_SAMPLE_DIR, "yes_no")
os.makedirs(YESNO_DATASET_PATH, exist_ok=True)

dataset = torchaudio.datasets.YESNO(YESNO_DATASET_PATH, download=True )

In [ ]:
i = 12
waveform, label, sample_rate = dataset[i]
print( f"wavform tensor shape: {waveform.shape} (datatype: {waveform.dtype}, device:{waveform.device})" )
print( f"sample rate: {sample_rate} Hz, label: {label}")

plot_specgram( waveform, sample_rate, title=f"Sample {i}: {label}" )
IPython.display.Audio(waveform, rate=sample_rate)


## Preparing data for training with DataLoaders

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=5, shuffle=True)

## Iterate through the DataLoader

In [ ]:
features, labels = next(iter( dataloader ) )
print(f"Feature batch shape: {features.size()}")
print(f"First sample label is: {labels[0]}")